# Systematic GO / pathway enrichment


In [ ]:
import os
import sys
import re
import gzip
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist
from statsmodels.stats.multitest import multipletests
import gseapy as gp

# local modules
sys.path.append(os.path.abspath("../scripts"))    # enrichment.py / plot_term_vs_lipid_with_arrows.py live in scripts/
from plot_term_vs_lipid_with_arrows import plot_term_vs_lipid
from enrichment import batch_enrich
from style import set_default_style, set_publication_style

# display / figure defaults
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
set_default_style()

## gene sets from CROP-seq differential analysis

From CROP_seq_LipoGrid_analyses.ipynb


In [ ]:
## differential expression results calculated in CROP_seq_LipoGrid_analyses.ipynb
all_results_df = pd.read_csv('../data/lipogrid/pilot/analysis/CROP_seq_log2FC_manw_pergeneKO_final.csv', index_col=0)
all_results_df

In [ ]:
# For each column ending with '_pvalue', perform FDR correction and insert a new column with '_FDR' immediately after
for col in list(all_results_df.columns):
    if col.endswith('_pvalue'):
        pvals = all_results_df[col].values
        _, fdr, _, _ = multipletests(pvals, method='fdr_bh')
        fdr_col = col.replace('_pvalue', '_FDR')
        # Insert FDR column immediately after pvalue column
        col_idx = all_results_df.columns.get_loc(col)
        all_results_df.insert(col_idx + 1, fdr_col, fdr)

# Filter out gRNAs to proceed with same gRNAs as in LipoGrid
with open('../data/lipogrid/pilot/analysis/final_4_runs/filtered_143target_genes.txt', 'r') as f:
    filtered_143target_genes = [line.strip() for line in f]


# keep columns in all_results_df if the name before _log2FC, _pvalue or _FDR is in filtered_143target_genes
filtered_columns = []
for col in all_results_df.columns:
    base_name = col.rsplit('_', 1)[0]  # Get the part before the last underscore
    if base_name in filtered_143target_genes:
        filtered_columns.append(col)

# subset all_results_df to only include filtered_columns
filtered_all_results_df = all_results_df[filtered_columns]
filtered_all_results_df

In [ ]:
## gene <-> cluster assignments, computed and saved by CROP_seq_analyses.ipynb (agglomerative
## clustering of the per-gene-KO log2FC matrix)
CLUSTER_DIR = Path('../data/lipogrid/pilot/analysis/CROP_seq')

target_gene_clusters = pd.read_csv(CLUSTER_DIR / 'target_genes_CROPseq-clusters.tsv', sep='\t')
gene_clustered = pd.read_csv(CLUSTER_DIR / 'genes_CROPseq_10cols_clusters.tsv', sep='\t')

## dict where for each cluster number, the corresponding genes are listed
gene_clustered_sets = gene_clustered.groupby('cluster')['gene'].apply(list).to_dict()

## 2. g:Profiler enrichment

`g_SCS` is g:Profiler's built-in correction (more conservative than BH). Set `background=` to your real universe (e.g. all expressed genes) for clean ORA.

In [ ]:
## optional: per-cluster g:Profiler enrichment (superseded below by the preranked GSEA on a
## custom lipid-focused GMT, which uses the full ranking rather than a hard gene-set cutoff)
RUN_CLUSTER_ENRICHMENT = False

if RUN_CLUSTER_ENRICHMENT:
    batchenr_clusters = batch_enrich(
        gene_clustered_sets,
        backend='gprofiler',
        organism='hsapiens',
        sources=['GO:BP', 'KEGG', 'REAC'],
        outdir='../data/lipogrid/pilot/analysis/CROP_seq/enrichments',
        top_n=5,
    )
    batchenr_clusters[['set', 'source', 'name', 'p_value', 'intersection_size', 'term_size']]

## 4. Preranked GSEA &mdash; for *ranked* gene lists

If you have a score per gene (log2 fold change from DESeq2/edgeR/limma, a t-statistic, signed `-log10(p)`, a feature-importance from a model, etc.), **don't threshold and run ORA** &mdash; you'll throw away most of the signal and the result will depend on an arbitrary cutoff.

Preranked GSEA walks down the *whole* ranked list and asks, for each pathway, whether its genes pile up at the top (positive NES) or bottom (negative NES). The output gives you:

- **NES** &mdash; normalized enrichment score (sign = direction, magnitude = strength)
- **NOM p-val / FDR q-val** &mdash; significance from gene-set permutations
- **Lead_genes** &mdash; the "leading edge" subset that drove the enrichment

In [ ]:
## create a custom GMT file with lipid-related terms from multiple databases, to use in future analyses
# 1. Inputs
GMT_FILES = {
    'HALLMARK': '../data/GSEA/hallmark.gmt',
    'REACTOME': '../data/GSEA/reactome.gmt',
    'KEGG':     '../data/GSEA/kegg.gmt',
    'GO_BP':    '../data/GSEA/go_bp.gmt',
    'C2CP':     '../data/GSEA/c2.cp.symbols.gmt',
    'C2CP_BIOCARTA': '../data/GSEA/c2.cp.biocarta.symbols.gmt',
    'C2CP_PID': '../data/GSEA/c2.cp.pid.symbols.gmt',
    'C2CP_WP': '../data/GSEA/c2.cp.wp.symbols.gmt',
    'C5_GO': '../data/GSEA/c5.go.v2024.1.Hs.symbols.gmt',
    'pathbank': '../data/GSEA/pathbank.Homo_sapiens.symbols.gmt'
}

OUT_PATH = '../data/GSEA/lipid_terms.pathways.gmt'

# 2. What counts as "lipid-related": case-insensitive whole-word(ish) substrings against the term name.
LIPID_KEYWORDS = [
    # generic
    r'lipid',
    r'lipo',                      # lipoprotein, lipogenesis, lipolysis
    r'fatty[_\s\-]?acid',
    r'\bfa\b',                    # rare, but appears in some KEGG terms
    # specific classes
    r'cholesterol', r'sterol', r'steroid',
    r'triglyceride', r'triacylglycerol', r'diacylglycerol', r'monoacylglycerol',
    r'phospholipid', r'glycerophospholipid', r'glycerolipid',
    r'sphingolipid', r'sphingomyelin', r'sphingosine',
    r'ceramide', r'cerebroside', r'ganglioside', r'glycolipid',
    r'phosphatidyl',              # PC, PE, PS, PI, PG
    r'cardiolipin',
    # transport / lipoproteins
    r'\bldl\b', r'\bhdl\b', r'\bvldl\b', r'chylomicron',
    r'apolipoprotein',
    # metabolism
    r'beta[_\s\-]?oxidation', r'β[_\s\-]?oxidation',
    r'fatty[_\s\-]?acyl',
    r'lipogenesis', r'lipolysis', r'adipogen', r'adipocyte',
    r'ketogenesis', r'ketone[_\s\-]?bod',
    r'mevalonate', r'isoprenoid', r'terpenoid',
    r'prostaglandin', r'eicosanoid', r'leukotriene', r'arachidonic',
    r'omega[_\s\-]?[36]',
    r'carnitine',
    # regulators that are almost always lipid context
    r'\bppar', r'srebp', r'scap',
]
LIPID_RE = re.compile('|'.join(LIPID_KEYWORDS), re.IGNORECASE)

# 2b. Words to EXCLUDE from matched terms (dropped if ANY of these match, case-insensitive)
EXCLUDE_KEYWORDS = [
    r'bile', r'desmosterolosis', r'hypercholesterolemia', r'hepatocytes', r'enterocyte', r'polymerase', r'Adipocyte', r'Familial', r'Ovarian',r'Adipogenesis',r'Huntingtons',
]
EXCLUDE_RE = re.compile('|'.join(EXCLUDE_KEYWORDS), re.IGNORECASE) \
             if EXCLUDE_KEYWORDS else None

# 3. Helpers
def read_gmt_with_desc(path):
    """Like read_gmt, but keeps each term's description column: [(term, desc, genes), ...]."""
    opener = gzip.open if str(path).endswith('.gz') else open
    out = []
    with opener(path, 'rt') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            term, desc, *genes = parts
            genes = [g for g in genes if g]
            out.append((term, desc, genes))
    return out

# 3b. Term canonicalization for cross-DB matching
# Patterns at the start of MSigDB / Enrichr term names that we want to peel off
DB_PREFIX_RE = re.compile(
    r'^('
    r'HALLMARK|REACTOME|KEGG(_LEGACY|_MEDICUS)?|WP|WIKIPATHWAYS|'
    r'BIOCARTA|PID|GOBP|GOMF|GOCC|GO_BP|GO_MF|GO_CC|MSIGDB'
    r')[_:\- ]+',
    re.IGNORECASE,
)
# Trailing IDs that some sources tack onto the term name
SUFFIX_ID_RE = re.compile(
    r'\s*('
    r'\(GO:\d+\)'                  # GO_BP_2023 style:  ... (GO:0019217)
    r'|R-HSA-\d+'                  # Reactome stable ID
    r'|Homo\s+sapiens\s+R-HSA-\d+' # Enrichr Reactome flavor
    r'|WP\d+'                      # WikiPathways  WP1234
    r'|hsa\d+'                     # KEGG  hsa00100
    r')\s*$',
    re.IGNORECASE,
)
NONALNUM_RE = re.compile(r'[^a-z0-9]+')

def canonical_key(term):
    """Lowercase, prefix/suffix-stripped, whitespace-collapsed key for grouping."""
    s = term.strip()
    # peel prefixes repeatedly (handles "REACTOME_KEGG_FOO" weirdness)
    while True:
        new = DB_PREFIX_RE.sub('', s)
        if new == s:
            break
        s = new
    s = SUFFIX_ID_RE.sub('', s).lower()
    s = NONALNUM_RE.sub(' ', s).strip()
    return s

def display_name(originals):
    """Pick a clean, human-readable name from the originals that share a key."""
    # Prefer entries that already have spaces and mixed case (likely Enrichr-style).
    has_space = [o for o in originals if ' ' in o and not o.isupper()]
    if has_space:
        return min(has_space, key=len)               # shortest readable variant
    # Otherwise reconstruct from the canonical key in Title Case.
    return canonical_key(originals[0]).title()

# 4. Filter, drop DB prefix, merge across DBs by canonical key
buckets = {}        # canonical_key -> {'genes': set, 'sources': set, 'origs': list}
hits_per_db = {}

for db, path in GMT_FILES.items():
    if not Path(path).exists():
        print(f'skip {db}: {path} not found')
        continue
    sets = read_gmt_with_desc(path)
    matched = [
        (t, d, g) for t, d, g in sets
        if LIPID_RE.search(t) and (EXCLUDE_RE is None or not EXCLUDE_RE.search(t))
    ]
    hits_per_db[db] = len(matched)

    for term, _desc, genes in matched:
        key = canonical_key(term)
        if not key:
            continue
        b = buckets.setdefault(
            key, {'genes': set(), 'sources': set(), 'origs': []}
        )
        b['genes'].update(genes)
        b['sources'].add(db)
        b['origs'].append(term)

print('matches per database:')
for db, n in hits_per_db.items():
    print(f'  {db:10s} {n}')
print(f'  unique canonical terms: {len(buckets)}')

# 5. Write merged GMT
written = {}     # display_name -> already used? (avoid accidental name clashes)

with open(OUT_PATH, 'w') as f:
    for key in sorted(buckets):
        info = buckets[key]
        name = display_name(info['origs'])
        # de-dup display names (rare, but possible after canonicalization)
        base = name; i = 2
        while name in written:
            name = f'{base} ({i})'; i += 1
        written[name] = True

        desc = ','.join(sorted(info['sources']))
        f.write('\t'.join([name, desc, *sorted(info['genes'])]) + '\n')

print(f'\nwrote {OUT_PATH}  ({len(buckets)} sets, '
      f'{sum(len(v["genes"]) for v in buckets.values())} gene entries)')

# 6. Diagnostic: which terms got merged across DBs?
merged = {k: v for k, v in buckets.items() if len(v['sources']) > 1}
print(f'\n{len(merged)} terms merged across multiple DBs (sample):')
for k, v in list(merged.items())[:20]:
    print(f"  [{','.join(sorted(v['sources']))}]  {display_name(v['origs'])}")
    for o in v['origs']:
        print(f"      <- {o}")

In [ ]:
# 1. GMT loader
def read_gmt(path):
    opener = gzip.open if path.endswith('.gz') else open
    d = {}
    with opener(path, 'rt') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            term, _desc, *genes = parts
            d[term] = [g for g in genes if g]
    return d


def make_ranking(df, gene, eps=1e-30):
    """Signed -log10(p) ranking for one KO. Returns a Series indexed by gene name."""
    lfc_col = f'{gene}_log2FC'
    p_col   = f'{gene}_pvalue'
    if lfc_col not in df.columns or p_col not in df.columns:
        raise KeyError(f'{gene}: need both {lfc_col} and {p_col}')
    s = np.sign(df[lfc_col]) * -np.log10(df[p_col].clip(lower=eps))
    return s.dropna().sort_values(ascending=False)


def discover_ko_genes(df, lfc_suffix='_log2FC', p_suffix='_pvalue'):
    """Any gene that has BOTH a <gene>_log2FC and a <gene>_pvalue column."""
    have_lfc = {c[:-len(lfc_suffix)] for c in df.columns if c.endswith(lfc_suffix)}
    have_p   = {c[:-len(p_suffix)]   for c in df.columns if c.endswith(p_suffix)}
    return sorted(have_lfc & have_p)

ko_genes = discover_ko_genes(filtered_all_results_df)
print(f'{len(ko_genes)} KO rankings: {ko_genes[:8]}{" ..." if len(ko_genes) > 8 else ""}')

rankings = {g: make_ranking(filtered_all_results_df, g) for g in ko_genes}

# Quick sanity check — sizes and a peek at the top of one ranking
sizes = pd.Series({g: len(s) for g, s in rankings.items()}).sort_values()
print('\nexample top of', ko_genes[0], ':')
print(rankings[ko_genes[0]].head())

In [ ]:
## Run batch GSEA enrichment analysis over all ranked lists

GMT_PATH = '../data/GSEA/lipid_terms.pathways.gmt'
gmt = read_gmt(GMT_PATH)
print(f'gene sets: {len(gmt)}')

# 1. Run prerank on each ranking
results = {}
tables  = []
for name, rnk in rankings.items():
    print(f'  prerank: {name}')
    pre = gp.prerank(
        rnk=rnk, gene_sets=gmt, min_size=5,
        threads=4, permutation_num=1000, seed=0,
        outdir=None, no_plot=True, verbose=False,
    )
    results[name] = pre
    df = pre.res2d.copy()
    df.insert(0, 'Ranking', name)
    tables.append(df)

all_results = pd.concat(tables, ignore_index=True)
all_results.to_csv(
    '../data/lipogrid/pilot/'
    'analysis/CROP_seq/enrichments/gsea_lipid_all_rankings_2_padj0.1.csv',
    index=False,
)

all_results = pd.read_csv('../data/lipogrid/pilot/analysis/CROP_seq/enrichments/gsea_lipid_all_rankings_2_padj0.1.csv')

# 2. Pivot to NES / FDR / NOM p-val matrices (terms x rankings) 
nes = all_results.pivot_table(index='Term', columns='Ranking',
                              values='NES', aggfunc='first')
fdr = all_results.pivot_table(index='Term', columns='Ranking',
                              values='FDR q-val', aggfunc='first')
pval = all_results.pivot_table(index='Term', columns='Ranking',
                               values='NOM p-val', aggfunc='first')

# 2b. BH-adjusted p-value per ranking
adj_pval = pval.copy().astype(float)
for col in pval.columns:
    col_vals = pval[col].values.astype(float)
    mask = ~np.isnan(col_vals)
    adj_col = np.ones_like(col_vals)
    if mask.sum() > 0:
        _, adj_col[mask], _, _ = multipletests(col_vals[mask], method='fdr_bh')
    adj_pval[col] = adj_col

# Keep only terms that are reasonably significant in at least one ranking,
# otherwise the heatmap is overwhelmed by noise.
TOP_N_PER_RANKING = 2
ADJ_PVAL_CUTOFF   = 0.1

top_terms = set()
for col in adj_pval.columns:
    sig = adj_pval[col][adj_pval[col] < ADJ_PVAL_CUTOFF].sort_values()
    top_terms.update(sig.head(TOP_N_PER_RANKING).index)

if not top_terms:
    print('No terms below adj. p-val cutoff -- relaxing to top-N regardless.')
    for col in nes.columns:
        top_terms.update(
            nes[col].abs().sort_values(ascending=False)
                   .head(TOP_N_PER_RANKING).index
        )

nes_top      = nes.loc[sorted(top_terms)].fillna(0)
fdr_top      = fdr.loc[sorted(top_terms)].fillna(1)
adj_pval_top = adj_pval.loc[sorted(top_terms)].fillna(1)

# Strip the DB__ prefix added by the lipid filter for cleaner labels
fdr_top.index      = nes_top.index
adj_pval_top.index = nes_top.index

# 3. Single combined figure: clustermap with adj. p-val stars
# Annotation based on BH-adjusted NOM p-val: * <.25, ** <.05, *** <.01
annot = (
    np.where(adj_pval_top.values < 0.01, '***',
    np.where(adj_pval_top.values < 0.05,  '**',
    np.where(adj_pval_top.values < 0.25,  '*',  '')))
)
annot = pd.DataFrame(annot, index=nes_top.index, columns=nes_top.columns)

vmax = max(abs(nes_top.values.min()), abs(nes_top.values.max()))
g = sns.clustermap(
    nes_top,
    cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
    annot=annot, fmt='', annot_kws={'fontsize': 8, 'color': 'black'},
    figsize=(max(6, 0.5 * nes_top.shape[1] + 4),
             max(6, 0.4 * nes_top.shape[0] + 2)),
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'NES'},
    dendrogram_ratio=(0.12, 0.08),
)
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
g.figure.suptitle('Lipid-pathway GSEA across rankings  '
                  '(* adj.p<.25, ** <.05, *** <.01)', y=1.02)
plt.show()

g.figure.savefig(
    '../data/lipogrid/pilot/'
    'analysis/CROP_seq/enrichments/gsea_lipid_clustermap_2_padj0.1.pdf',
    dpi=200, bbox_inches='tight',
)

In [ ]:
# NES remains quite high even if there is no enrichments at all (p-val close to 1, hence shrink values that have high p-values to better visualize the actual enriched signals)

def shrink_nes(
    nes: pd.DataFrame,
    fdr: pd.DataFrame,
    method: str = "sigmoid",
    threshold: float = 0.25,
    scale: float = 0.05,
    k: float = 1.0,
) -> pd.DataFrame:
    """
    Element-wise shrinkage of an NES matrix using the matched FDR matrix.

    Both `nes` and `fdr` are terms × rankings DataFrames with identical
    shape, index, and columns (e.g. the `nes_top` / `fdr_top` produced by
    the GSEA pivot above).

    Per-cell weight w(FDR) ∈ [0, 1]:

      'sigmoid' : w = 1 / (1 + exp((FDR - threshold) / scale))
                  Sharp transition centred at `threshold`. Significant cells
                  (FDR << threshold) are essentially unchanged; non-sig
                  cells (FDR >> threshold) are pushed to ~0.
      'power'   : w = (1 - FDR) ** k
                  Smooth. k > 1 protects significant cells better; k = 0.5–1
                  is mild, k = 4 is fairly aggressive on insignificant cells.
      'linear'  : w = 1 - FDR     (== 'power' with k = 1).
      'hard'    : w = (FDR <= threshold).astype(float)
                  Sets NES = 0 above the cutoff (standard GSEA convention).

    Returns
    -------
    nes_shrunk : pd.DataFrame
        Same shape / index / columns as `nes`, with element-wise shrunk values.
    """
    fdr_arr = (
        fdr.reindex(index=nes.index, columns=nes.columns)
           .astype(float)
           .clip(lower=0, upper=1)
    )

    if method == "sigmoid":
        w = 1.0 / (1.0 + np.exp((fdr_arr - threshold) / scale))
    elif method == "power":
        w = (1.0 - fdr_arr) ** k
    elif method == "linear":
        w = 1.0 - fdr_arr
    elif method == "hard":
        w = (fdr_arr <= threshold).astype(float)
    else:
        raise ValueError(f"unknown method {method!r}")

    return nes.astype(float) * w

In [ ]:
nes_top_shrunk = shrink_nes(nes_top, adj_pval_top, method="sigmoid",
                            threshold=0.9, scale=0.1)

In [ ]:
## GSEA penalizes small gene sets, so might be that we miss some biology if it only involves a small set of genes that are upregulated, but its the most statistical sound way
nes_top.index        = nes_top.index.str.split('%').str[0].str.strip()
fdr_top.index        = fdr_top.index.str.split('%').str[0].str.strip()
adj_pval_top.index   = adj_pval_top.index.str.split('%').str[0].str.strip()
nes_top_shrunk.index = nes_top_shrunk.index.str.split('%').str[0].str.strip()

for df in (nes_top, fdr_top, adj_pval_top, nes_top_shrunk):
    df.index = (df.index
                .str.replace(r'R-HSA-\d+', '', regex=True)   # drop the Reactome tag
                .str.replace(r'\(GO:\d+\)', '', regex=True)
                .str.replace(r'\s+', ' ', regex=True)        # collapse double spaces
                .str.strip())

# Drop duplicate term names produced by the prefix stripping above,
# keeping the row with the highest max(|NES|) within each duplicate group.
abs_max = nes_top.abs().max(axis=1).to_numpy()
idx_arr = nes_top.index.to_numpy()
keep = np.zeros(len(nes_top), dtype=bool)
for label in pd.unique(idx_arr):
    pos = np.flatnonzero(idx_arr == label)
    keep[pos[np.argmax(abs_max[pos])]] = True
nes_top, fdr_top, adj_pval_top, nes_top_shrunk = (
    nes_top[keep], fdr_top[keep], adj_pval_top[keep], nes_top_shrunk[keep]
)

In [ ]:
nes_top_shrunk.to_csv("../data/lipogrid/pilot/analysis/CROP_seq/enrichments/nes_top_shrunk.tsv", sep="\t")
nes_top_shrunk


In [ ]:
def deduplicate_correlated_terms(
    nes: pd.DataFrame,
    corr_threshold: float = 0.9,
    name_jaccard_threshold: float | None = 0.4,
    high_corr_threshold: float | None = None,
    high_corr_name_jaccard_threshold: float | None = None,
    method: str = "spearman",
    score: str = "max_abs",
    use_abs_corr: bool = False,
    must_keep: list[str] | None = None,
):
    """
    Collapse highly-correlated, similarly-named rows of a NES matrix,
    keeping the term with the strongest signal per cluster.

    Two-tier merging:
      - pairs with edge_strength >= corr_threshold use name_jaccard_threshold
      - pairs with edge_strength >= high_corr_threshold use
        high_corr_name_jaccard_threshold instead (e.g. relax the name
        requirement when the correlation alone is very strong, or set the
        high-tier Jaccard to None to skip the name check entirely).

    Step 0 collapses *exact* duplicate index labels first by keeping the
    row with the highest max(|NES|) within each duplicate group. Step 1+
    then performs the correlation/name clustering on a unique-index frame.

    must_keep: terms that are always retained as cluster representatives,
    even if they would normally be merged away. If multiple must_keep terms
    fall in the same cluster, all of them are kept.
    """
    # validate the two-tier config
    if high_corr_threshold is not None and high_corr_threshold < corr_threshold:
        raise ValueError(
            "high_corr_threshold must be >= corr_threshold "
            f"(got {high_corr_threshold} < {corr_threshold})."
        )

    must_keep_set = set(must_keep) if must_keep else set()

    # 0. collapse exact-duplicate index labels, keep the row with the
    #    highest max(|NES|) per duplicate group.
    if not nes.index.is_unique:
        abs_max_per_row = nes.abs().max(axis=1).to_numpy()
        index_arr = nes.index.to_numpy()
        keep_mask = np.zeros(len(nes), dtype=bool)
        for label in pd.unique(index_arr):
            positions = np.flatnonzero(index_arr == label)
            if len(positions) == 1:
                keep_mask[positions[0]] = True
            else:
                keep_mask[positions[np.argmax(abs_max_per_row[positions])]] = True
        nes = nes.iloc[keep_mask].copy()

    # 1. row-by-row correlation
    corr = nes.T.corr(method=method).values
    np.fill_diagonal(corr, 0)
    edge_strength = np.abs(corr) if use_abs_corr else corr

    terms = list(nes.index)
    n = len(terms)

    # 2. optional name-token Jaccard (built if either tier needs it)
    needs_tokens = (
        name_jaccard_threshold is not None
        or (high_corr_threshold is not None
            and high_corr_name_jaccard_threshold is not None)
    )
    toks = [set(str(t).lower().split()) for t in terms] if needs_tokens else None

    # 3. union-find clustering
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj

    for i in range(n):
        for j in range(i + 1, n):
            s = edge_strength[i, j]
            if s < corr_threshold:
                continue

            # decide which Jaccard threshold applies to this pair
            if high_corr_threshold is not None and s >= high_corr_threshold:
                active_jacc_threshold = high_corr_name_jaccard_threshold
            else:
                active_jacc_threshold = name_jaccard_threshold

            if active_jacc_threshold is not None:
                a, b = toks[i], toks[j]
                jacc = len(a & b) / len(a | b) if (a | b) else 0.0
                if jacc < active_jacc_threshold:
                    continue

            union(i, j)

    # 4. score every term
    if score == "max_abs":
        scores = nes.abs().max(axis=1)
    elif score == "sum_abs":
        scores = nes.abs().sum(axis=1)
    elif score == "range":
        scores = nes.max(axis=1) - nes.min(axis=1)
    elif score == "l2":
        scores = np.sqrt((nes ** 2).sum(axis=1))
    else:
        raise ValueError(f"Unknown score: {score!r}")

    # 5. pick the strongest term per cluster; must_keep terms always survive
    clusters = {}
    for idx in range(n):
        clusters.setdefault(find(idx), []).append(idx)

    keep, cluster_map = [], {}
    for members in clusters.values():
        forced = [i for i in members if terms[i] in must_keep_set]

        if forced:
            # All must_keep terms are retained; non-forced members are
            # attributed to the highest-scoring forced term.
            best_forced = max(forced, key=lambda i: scores.iloc[i])
            non_forced = [i for i in members if terms[i] not in must_keep_set]
            for fi in forced:
                keep.append(fi)
                owned = [terms[fi]]
                if fi == best_forced:
                    owned += [terms[m] for m in non_forced]
                cluster_map[terms[fi]] = owned
        else:
            best = max(members, key=lambda i: scores.iloc[i])
            keep.append(best)
            cluster_map[terms[best]] = [terms[m] for m in members]

    keep.sort()
    filtered = nes.iloc[keep].copy()
    return filtered, cluster_map

In [ ]:
nes_matrix = nes_top_shrunk

filtered_nes, merge_log = deduplicate_correlated_terms(
    nes_matrix,
    corr_threshold=0.6,
    name_jaccard_threshold=0.5,
    high_corr_threshold=0.8, high_corr_name_jaccard_threshold=0.1,
    score="sum_abs",
    must_keep=["Cholesterol Biosynthesis", "Fatty Acid Biosynthetic Process"],
)
fdr_top_filtered     = fdr_top.loc[filtered_nes.index].copy()
adj_pval_top_filtered = adj_pval_top.loc[filtered_nes.index].copy()

print(f"Reduced {nes_matrix.shape[0]} -> {filtered_nes.shape[0]} terms")

# Inspect merges
for kept, members in merge_log.items():
    if len(members) > 1:
        print(f"\nKept: {kept}  (|NES|max = {nes_matrix.loc[kept].abs().max():.2f})")
        for m in members:
            if m != kept:
                print(f"   merged: {m}")

In [ ]:
filtered_nes.to_csv("../data/lipogrid/pilot/analysis/CROP_seq/enrichments/filtered_nes.csv")

In [ ]:
# 3. Single combined figure: clustermap with FDR / adj. p-val stars
# Toggle: BH-adjusted NOM p-val (adj_pval_top_filtered) or GSEA FDR q-val (fdr_top_filtered)
sig_matrix = adj_pval_top_filtered   # or: fdr_top_filtered

annot = (
    np.where(sig_matrix.values < 0.01, '***',
    np.where(sig_matrix.values < 0.05,  '**',
    np.where(sig_matrix.values < 0.25,  '*',  '')))
)
annot = pd.DataFrame(annot, index=filtered_nes.index, columns=filtered_nes.columns)

vmax = max(abs(filtered_nes.values.min()), abs(filtered_nes.values.max()))

# 1. Distance: correlation (1 - Pearson) — groups by enrichment pattern, not magnitude
row_dist = pdist(filtered_nes.values,         metric='euclidean')
col_dist = pdist(filtered_nes.values.T,       metric='euclidean')

# 2. Linkage: 'average' is robust; 'complete' makes tighter, more separated clusters
row_link = linkage(row_dist, method='average')
col_link = linkage(col_dist, method='average')

# 3. Optimal leaf ordering — reorders dendrogram leaves to minimise sum of adjacent
#    distances. Pure cosmetic improvement; doesn't change the clustering structure.
row_link = optimal_leaf_ordering(row_link, row_dist)
col_link = optimal_leaf_ordering(col_link, col_dist)

g = sns.clustermap(
    filtered_nes,
    row_linkage=row_link,
    col_linkage=col_link,
    cmap='PuOr_r', center=0, vmin=-vmax, vmax=vmax,
    annot=annot, fmt='', annot_kws={'fontsize': 15, 'color': 'black'},
    figsize=(max(6, 0.5 * filtered_nes.shape[1] + 4),
             max(6, 0.4 * filtered_nes.shape[0] + 2)),
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'NES'},
    dendrogram_ratio=(0.001, 0.001),
)
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
g.cax.set_position([1.03, .40, .015, .15])
g.cax.tick_params(labelsize=30)
g.cax.set_ylabel('NES', fontsize=34, fontweight='bold')
g.ax_heatmap.tick_params(axis='x', labelsize=22)
g.ax_heatmap.tick_params(axis='y', labelsize=24)
# save as pdf
g.figure.savefig(
    '../data/lipogrid/pilot/analysis/CROP_seq/enrichments/gsea_lipid_clustermap_mergedterms_collapse.pdf',
    bbox_inches='tight')

plt.show()

In [ ]:
set_publication_style()

# selections (rows = terms, cols = genes)
genes_to_label = ['ALDH9A1','APOE','B4GALT6','SREBF1','SREBF2','NPC1','NFYA',
                  'HMGCS1','HSD17B12','HADHB','FASN','ACOT4','HMGCR','LAMTOR1','CDIPT']
terms_to_label = ["Unsaturated Fatty Acid Metabolic Process","Cholesterol Metabolism With Bloch And Kandutschrussell Pathways",
                  "Isoprenoid Binding","Ceramide Catabolic Process","Glycosphingolipid Binding","Terpenoid Biosynthetic Process",
                  "Cellular Lipid Catabolic Process","Fatty Acid Oxidation",
                  "Cellular Response To Sterol","Mitochondrial Fatty Acid Synthesis And Respiration","Chylomicron","Cholesterol Efflux",
                  "Phosphatidylinositol Acyl-Chain Remodeling","Prostaglandin Signaling","Lipoprotein Catabolic Process",
                  "Neolacto Series Sphingolipid Metabolism","Arachidonic Acid Metabolism","Steroid Hormone Secretion",
                  "Cholesterol Biosynthesis","Fatty Acid Biosynthetic Process"]
term_rename = {
    "Cholesterol Metabolism With Bloch And Kandutschrussell Pathways": "Cholesterol Metabolism",
    "Negative Regulation Of Lipid Kinase Activity": "Neg. Reg. Lipid Kinase",
    "Mitochondrial Fatty Acid Synthesis And Respiration": "Mito. FA Synthesis & Resp.",
    "Unsaturated Fatty Acid Metabolic Process": "Unsat FA metabolic Process",
    "Phosphatidylinositol Acyl-Chain Remodeling": "PI Acyl-Chain Remodeling",
    "Neolacto Series Sphingolipid Metabolism": "Neolacto Sphingolipid Metabolism",
}

# subset to selected rows/cols (report anything missing)
miss_t = [t for t in terms_to_label if t not in filtered_nes.index]
miss_g = [gn for gn in genes_to_label if gn not in filtered_nes.columns]
if miss_t: print(f"[labels] terms not in matrix: {sorted(miss_t)}")
if miss_g: print(f"[labels] genes not in matrix: {sorted(miss_g)}")
rows = [t for t in terms_to_label if t in filtered_nes.index]
cols = [gn for gn in genes_to_label if gn in filtered_nes.columns]
nes = filtered_nes.loc[rows, cols]
sig = sig_matrix.loc[rows, cols]                       # sig_matrix = adj_pval_top_filtered or fdr_top_filtered

annot = pd.DataFrame(
    np.where(sig.values < 0.01, '***', np.where(sig.values < 0.05, '**', np.where(sig.values < 0.25, '*', ''))),
    index=nes.index, columns=nes.columns)

vmax = np.nanmax(np.abs(nes.values))

# cluster the subset
rd = pdist(nes.values, 'euclidean');   cd = pdist(nes.values.T, 'euclidean')
rl = optimal_leaf_ordering(linkage(rd, 'average'), rd)
cl = optimal_leaf_ordering(linkage(cd, 'average'), cd)

g = sns.clustermap(
    nes, row_linkage=rl, col_linkage=cl,
    cmap='PuOr_r', center=0, vmin=-vmax, vmax=vmax,
    annot=annot, fmt='', annot_kws={'fontsize': 11, 'color': 'black'},
    figsize=(0.25*nes.shape[1] + 6, 0.25*nes.shape[0] + 3),
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'NES'}, dendrogram_ratio=(0.001, 0.001),
)
g.ax_row_dendrogram.set_visible(False); g.ax_col_dendrogram.set_visible(False)
g.ax_heatmap.set_xlabel(''); g.ax_heatmap.set_ylabel('')

# rename term (y) labels; all shown since only selected rows remain
g.ax_heatmap.set_yticklabels([term_rename.get(t.get_text(), t.get_text())
                              for t in g.ax_heatmap.get_yticklabels()], fontsize=12)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=12)
g.ax_heatmap.tick_params(length=0)
for s in g.ax_heatmap.spines.values(): s.set_visible(True); s.set_linewidth(1)

g.cax.set_position([0.84, 0.35, 0.02, 0.3])
g.cax.set_ylabel('NES', fontsize=13, fontweight='bold')
g.cax.tick_params(labelsize=11, length=5, width=1)
for s in g.cax.spines.values():
    s.set_linewidth(1)

g.figure.savefig("../data/lipogrid/pilot/analysis/internalnorm_finalfigs/gsea_lipid_clustermap_selection_compact.pdf",
                 dpi=300, bbox_inches="tight")
plt.show()

### compare certain metabolic processes with the actual changes in lipid concentration in the cells

In [ ]:
## read in processed file with lipids log2FC over intergenic controls
lipids_results_df = pd.read_csv('../data/lipogrid/pilot/analysis/final_4_runs/all_lipid_log2FC_pvalues_per_gene.csv', index_col=0)
lipids_results_df = lipids_results_df.loc[:, ~lipids_results_df.columns.str.startswith('Intergenic')]## extract log2FC table
lipids_logFC_df = lipids_results_df.loc[:, lipids_results_df.columns.str.endswith('_log2FC')]      # keep matching cols
lipids_logFC_df.columns = lipids_logFC_df.columns.str.removesuffix('_log2FC')     # strip the suffix
## extract Pvalue table
lipids_pval_df = lipids_results_df.loc[:, lipids_results_df.columns.str.endswith('_pvalue')]      # keep matching cols
lipids_pval_df.columns = lipids_pval_df.columns.str.removesuffix('_pvalue')     # strip the suffix
lipids_pval_df

In [ ]:
# 1. Paths
DATA_DIR = Path("../data/lipogrid/pilot/analysis")

LIPID_CSV   = DATA_DIR / "final_4_runs/all_lipid_log2FC_pvalues_per_gene.csv"
CROPSEQ_CSV = DATA_DIR / "CROP_seq_log2FC_manw_pergeneKO_sameKOs.csv"
FIG_OUT     = DATA_DIR / "CROP_seq/enrichments/example_plot_with_arrows.pdf"

# 2. Lipid log2FC and p-value matrices

lipid_raw = pd.read_csv(LIPID_CSV, index_col=0)

log2fc_cols = [c for c in lipid_raw.columns if c.endswith("_log2FC")]
pval_cols   = [c for c in lipid_raw.columns if c.endswith("_pvalue")]

lipid_df = lipid_raw[log2fc_cols].copy()
lipid_df.columns = [c.removesuffix("_log2FC") for c in log2fc_cols]

lipid_pval_df = lipid_raw[pval_cols].copy()
lipid_pval_df.columns = [c.removesuffix("_pvalue") for c in pval_cols]

# Sanity: both matrices should now share the same KO gene columns.
assert (lipid_df.columns == lipid_pval_df.columns).all()

# 3. CROP-seq RNA effect matrix  (left in its native wide format —
#    the plot function expects "<KO>_log2FC" / "<KO>_pvalue" columns).
ko_effect_df = pd.read_csv(CROPSEQ_CSV, index_col=0)

TERM  = "Cholesterol Biosynthesis"   # must exist in nes_df.index
LIPID = "Cholesterol"                     # must exist in lipid_df.index
FIG_OUT = DATA_DIR / "CROP_seq/enrichments" / f"{TERM.replace(' ', '_')}_{LIPID.replace(' ', '_')}.pdf"
#TERM  = "Fatty Acid Biosynthetic Process"   # must exist in nes_df.index LIPID = "PC 32:1"  
#LIPID = "PC 32:1"                     # must exist in lipid_df.index

fig, ax = plot_term_vs_lipid(
    nes_df=nes_top_shrunk,
    lipid_df=lipid_df,
    term=TERM,
    lipid=LIPID,

    # Significance overlays for point coloring
    fdr_df=adj_pval_top,           fdr_cutoff=0.1,
    lipid_pval_df=lipid_pval_df, lipid_pval_cutoff=0.01,

    # Labels
    annotate_top_n=40,
    extra_genes=["ACAT2", "HMGCS1", "HMGCR", "FDFT1", "DHCR7","INSIG1", "SREBF2"],   # give specific genes another label
    show_nonsig=False,

    # KO → target arrow overlay
    ko_effect_df=ko_effect_df,
    ko_pval_cutoff=0.01,
    ko_log2fc_min=0.2,           # set e.g. 0.1 to require |log2FC| >= 0.1
    arrow_up_color="#0CC828",    #   = KO upregulates the target
    arrow_down_color="#B9291C",  #   = KO downregulates the target
    arrow_alpha=0.6,
    arrow_lw=1.2,
    arrow_curve_rad=0.15,        # 0 = straight; >0 curves reciprocal pairs
    arrow_mutation_scale=14,     # arrowhead size
    arrow_in_legend=True,

    figsize=(10, 8),
)

fig.savefig(FIG_OUT, dpi=200, bbox_inches="tight")
print(f"Saved: {FIG_OUT}")
plt.rcParams["pdf.fonttype"] = 42  # editable, embeddable text
plt.savefig("../data/lipogrid/pilot/analysis/internalnorm_finalfigs/CholBiosynth_cholesterol_connections.pdf", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
nes_top_shrunk.to_csv(DATA_DIR / "CROP_seq/enrichments/nes_top_shrunk.tsv", sep="\t")
nes_top_shrunk